# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Display dataset name and description
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Review available record sets, fields, and their `@id` values. This helps understand the structural composition and content organization of the dataset as defined in its Croissant schema.

In [ ]:
# List available record sets in the dataset (referenced by their @id)
print("Available Record Sets (by @id):")
for recordset in metadata.record_sets:
    print(f"- @id: {recordset['@id']}   name: {recordset.get('name','')}    description: {recordset.get('description','')}")

# For each record set, list the available fields (by @id) and columns
for rs in metadata.record_sets:
    print(f"\nFields for RecordSet @id: {rs['@id']} ({rs.get('name','')})")
    fields = rs.get('fields', [])
    for field in fields:
        fieldid = field.get('@id') or field
        print(f"  Field @id: {fieldid}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use record set and field `@id`s obtained above.

In [ ]:
# Identify all record set @ids
record_set_ids = [rs['@id'] for rs in metadata.record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records from record set @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Columns: {df.columns.tolist()}")
    display(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Below, we demonstrate filtering on a numeric field, normalizing, and grouping. Please change the `numeric_field_id` and `group_field_id` to valid field `@id`s that exist in the columns for your record set.

In [ ]:
# -- Parameters: Select record set and field '@id's to analyze --
# Fill in the actual record set @id and field @ids as found in section 2

# Example (update as per your dataset):
main_recordset_id = record_set_ids[0]  # Use the first record set, or replace with desired one

# If you know the actual @id field for a numeric field, set it below
numeric_field_id = None
# Identify a numeric column to use
print(f"Numeric sample columns in {main_recordset_id}:")
numeric_candidates = []
for col in dataframes[main_recordset_id].columns:
    if pd.api.types.is_numeric_dtype(dataframes[main_recordset_id][col]):
        numeric_candidates.append(col)
print(numeric_candidates)
if numeric_candidates:
    numeric_field_id = numeric_candidates[0]   # Use first numeric field
else:
    raise Exception('No numeric field found for EDA.')

threshold = dataframes[main_recordset_id][numeric_field_id].mean()  # Use mean as an example threshold
filtered_df = dataframes[main_recordset_id][dataframes[main_recordset_id][numeric_field_id] > threshold].copy()
print(f"Filtered records in {main_recordset_id} with {numeric_field_id} > {threshold}:")
display(filtered_df.head())

# Normalize the numeric field
norm_col = f"{numeric_field_id}_normalized"
filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, norm_col]].head())

# Try grouping by a categorical field if available
group_field_id = None
non_numeric_cols = [col for col in dataframes[main_recordset_id].columns if not pd.api.types.is_numeric_dtype(dataframes[main_recordset_id][col])]
if non_numeric_cols:
    group_field_id = non_numeric_cols[0]
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped data by {group_field_id} (mean of {numeric_field_id}):")
    display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using matplotlib or seaborn.

In [ ]:
# Example visualization: Histogram of the chosen numeric field
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,5))
sns.histplot(dataframes[main_recordset_id][numeric_field_id].dropna(), kde=True)
plt.title(f"Distribution of {numeric_field_id} in RecordSet {main_recordset_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# Boxplot of numeric_field_id grouped by group_field_id (if it exists)
if group_field_id:
    plt.figure(figsize=(10, 6))
    sns.boxplot(data=dataframes[main_recordset_id], x=group_field_id, y=numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id} (RecordSet {main_recordset_id})")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
This notebook demonstrated how to use `mlcroissant` to load and explore a FAIR dataset in Croissant format. You can extend the analysis and visualizations to fit your scientific goals. Please refer to the dataset and Croissant documentation for further details about the record sets, fields, and their meanings.